# Baseline Single-Task — Dataset 4 : CWRU Bearing Fault

| Champ | Valeur |
|-------|--------|
| Dataset | CWRU Bearing Fault (Case Western Reserve University) |
| Scénario | `no_split` — toutes les conditions, une seule tâche |
| Features | 9 features temporelles (RMS, kurtosis, skewness, crest factor, etc.) |
| Label | Binaire (healthy / fault) |
| Modèles | EWC · HDC · TinyOL · KMeans · Mahalanobis · DBSCAN |
| Expériences | exp_068 – exp_073 |
| Sprint | 12 — S12-09 |

**Objectif** : Établir la performance maximale de chaque modèle en l'absence de toute contrainte CL.
Ce notebook est la référence absolue (*plafond*) pour mesurer le coût du continual learning
dans les scénarios `by_fault_type` (exp_074–079) et `by_severity` (exp_080–085).

> `FIXME(gap1)` : Si AUC-ROC ≈ 0.5 sans CL, le label binaire ou les features sont inadaptés.
> Voir la discussion Section 6.

```bash
jupyter nbconvert --to notebook --execute \
    notebooks/cl_eval/baselines/cwru_single_task.ipynb \
    --output /tmp/cwru_single_task_executed.ipynb
```

In [ ]:
# Section 1 — Setup & imports
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

np.random.seed(42)

# --- CWD navigation : notebook 3 niveaux de profondeur ---
_cwd = Path(".").resolve()
if _cwd.name == "baselines":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import save_figure

EXP_DIRS = {
    "EWC":         Path("experiments/exp_068_ewc_cwru_single_task"),
    "HDC":         Path("experiments/exp_069_hdc_cwru_single_task"),
    "TinyOL":      Path("experiments/exp_070_tinyol_cwru_single_task"),
    "KMeans":      Path("experiments/exp_071_kmeans_cwru_single_task"),
    "Mahalanobis": Path("experiments/exp_072_mahalanobis_cwru_single_task"),
    "DBSCAN":      Path("experiments/exp_073_dbscan_cwru_single_task"),
}
FIGURES_DIR = REPO_ROOT / "notebooks/figures/cl_evaluation/baselines/cwru"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_MODELS   = ["EWC", "HDC", "TinyOL"]
UNSUPERVISED_MODELS = ["KMeans", "Mahalanobis", "DBSCAN"]
MODEL_ORDER = SUPERVISED_MODELS + UNSUPERVISED_MODELS
CWRU_N_FEATURES = 9

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"FIGURES_DIR : {FIGURES_DIR}")
print("\nExpériences disponibles :")
for name, path in EXP_DIRS.items():
    p = path / "results" / "metrics_single_task.json"
    status = "OK" if p.exists() else "MANQUANTE (mock activé)"
    print(f"  {name:15s}: {status}")

In [ ]:
# Section 1 (suite) — Chargement des résultats (fallback mock par modèle)

# Valeurs mock calées sur les résultats réels CWRU (exp_068–073)
MOCK_DATA: dict[str, dict] = {
    "EWC": {
        "accuracy": 0.9783, "f1": 0.9880, "auc_roc": 0.9957,
        "ram_peak_bytes": 1171, "inference_latency_ms": 0.020, "n_params": 865,
    },
    "HDC": {
        "accuracy": 0.8870, "f1": 0.9330, "auc_roc": 0.9372,
        "ram_peak_bytes": 7920, "inference_latency_ms": 0.048, "n_params": 1024,
    },
    "TinyOL": {
        "accuracy": 0.9000, "f1": 0.9474, "auc_roc": 0.8773,
        "ram_peak_bytes": 944, "inference_latency_ms": 0.005, "n_params": 397,
    },
    "KMeans": {
        "accuracy": 0.1587, "f1": 0.1224, "auc_roc": 0.6015,
        "ram_peak_bytes": 5386, "inference_latency_ms": 0.194, "n_params": 18,
    },
    "Mahalanobis": {
        "accuracy": 0.1391, "f1": 0.0833, "auc_roc": 0.5479,
        "ram_peak_bytes": 1644, "inference_latency_ms": 0.005, "n_params": 90,
    },
    "DBSCAN": {
        "accuracy": 0.1457, "f1": 0.0966, "auc_roc": 0.8416,
        "ram_peak_bytes": 118468, "inference_latency_ms": 0.232, "n_params": 11286,
    },
}

results: dict[str, dict] = {}
mock_flags: dict[str, bool] = {}

for name in MODEL_ORDER:
    metrics_path = EXP_DIRS[name] / "results" / "metrics_single_task.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            data = json.load(f)
        mock_flags[name] = False
        # Normalisation : acc_final -> accuracy, f1_score -> f1 (KMeans, Mahalanobis, DBSCAN)
        if "accuracy" not in data and "acc_final" in data:
            data["accuracy"] = data["acc_final"]
        if "f1" not in data and "f1_score" in data:
            data["f1"] = data["f1_score"]
        results[name] = data
        print(f"[OK]   {name}: chargé depuis {metrics_path}")
    else:
        mock_flags[name] = True
        results[name] = MOCK_DATA[name].copy()
        print(f"[MOCK] {name}: exp manquante — valeurs fictives utilisées")

# Injection n_macs — coût d'inférence portable
from src.evaluation import compute_macs

CWRU_MACS_ARGS: dict[str, dict] = {
    "EWC":         dict(n_features=CWRU_N_FEATURES, hidden_dims=[32, 16], n_classes=1),
    "TinyOL":      dict(n_features=CWRU_N_FEATURES, encoder_dims=[16, 12, 8], n_classes=1),
    "HDC":         dict(n_features=CWRU_N_FEATURES, dim_hv=1024, n_classes=2),
    "KMeans":      dict(n_features=CWRU_N_FEATURES, n_clusters=max(1, results["KMeans"].get("n_params", 18) // CWRU_N_FEATURES)),
    "Mahalanobis": dict(n_features=CWRU_N_FEATURES),
    "DBSCAN":      dict(n_features=CWRU_N_FEATURES, n_core_samples=max(1, results["DBSCAN"].get("n_params", 11286) // CWRU_N_FEATURES)),
}
for name in MODEL_ORDER:
    results[name]["n_macs"] = compute_macs(name, **CWRU_MACS_ARGS[name])

IS_ANY_MOCK = any(mock_flags.values())
if IS_ANY_MOCK:
    mocked = [n for n, v in mock_flags.items() if v]
    display(Markdown(
        f"> ⚠️ **MOCK DATA** — Les résultats suivants sont fictifs pour : "
        f"**{', '.join(mocked)}**. Lancer les expériences exp_068–073 pour les résultats réels."
    ))

all_accs = [results[n]["accuracy"] for n in MODEL_ORDER]
ALL_NEAR_RANDOM = all(abs(a - 0.5) < 0.12 for a in all_accs)
if ALL_NEAR_RANDOM:
    display(Markdown(
        "> ⚠️ **DIAGNOSTIC STRUCTUREL** — Toutes les accuracies sont proches de 0.50 "
        "(même sans contrainte CL). Le label est peut-être trop bruité. "
        "Voir `FIXME(gap1)` dans la Section 6."
    ))

In [ ]:
# Section 2 — Tableau comparatif global

def _fmt(val, fmt: str = "") -> str:
    if val is None:
        return "—"
    try:
        if np.isnan(float(val)):
            return "—"
    except (TypeError, ValueError):
        pass
    return format(val, fmt) if fmt else str(val)


def _fmt_macs(n: int) -> str:
    if n >= 1_000_000:
        return f"{n / 1_000_000:.2f} M"
    if n >= 1_000:
        return f"{n / 1_000:.1f} k"
    return str(n)


rows = []
for name in MODEL_ORDER:
    r = results[name]
    ram_kb = r.get("ram_peak_bytes", 0) / 1024
    in_budget = "✓" if ram_kb <= 64.0 else "✗ hors budget"
    rows.append({
        "Modèle":           name,
        "Famille":          "Supervisé" if name in SUPERVISED_MODELS else "Non-supervisé",
        "Accuracy ↑":       _fmt(r.get("accuracy"), ".4f"),
        "F1 ↑":             _fmt(r.get("f1"), ".4f"),
        "AUC-ROC ↑":        _fmt(r.get("auc_roc"), ".4f"),
        "RAM peak (Ko) ↓":  _fmt(ram_kb, ".1f"),
        "STM32 ≤ 64 Ko":    in_budget,
        "MACs ↓":           _fmt_macs(r.get("n_macs", 0)),
        "Latence (ms) ↓":   _fmt(r.get("inference_latency_ms"), ".3f"),
        "Params":           f"{r.get('n_params', 0):,}",
    })

rows.append({
    "Modèle":           "Best CL scenario (à compléter)",
    "Famille":          "—",
    "Accuracy ↑":       "—",
    "F1 ↑":             "—",
    "AUC-ROC ↑":        "—",
    "RAM peak (Ko) ↓":  "—",
    "STM32 ≤ 64 Ko":    "—",
    "MACs ↓":           "—",
    "Latence (ms) ↓":   "—",
    "Params":           "—",
})

df = pd.DataFrame(rows).set_index("Modèle")
df_sorted = df.iloc[:-1].sort_values("Accuracy ↑", ascending=False)
df_display = pd.concat([df_sorted, df.iloc[[-1]]])

display(Markdown("### Tableau comparatif — Baseline Single-Task (Dataset 4 CWRU)"))
if IS_ANY_MOCK:
    display(Markdown("*⚠️ Données partiellement ou totalement mock — voir Section 1.*"))
display(df_display)

In [ ]:
# Section 3 — Barplot groupé Accuracy + F1 (supervisés vs non-supervisés)

COLORS = {
    "EWC":         "#1f77b4",
    "HDC":         "#4a90d9",
    "TinyOL":      "#7eb8e8",
    "KMeans":      "#2ca02c",
    "Mahalanobis": "#5cc05c",
    "DBSCAN":      "#98df8a",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (metric_key, metric_label) in enumerate([("accuracy", "Accuracy"), ("f1", "F1")]):
    ax = axes[ax_idx]
    x_pos = np.arange(len(MODEL_ORDER))
    vals = [results[name][metric_key] for name in MODEL_ORDER]
    bar_colors = [COLORS[name] for name in MODEL_ORDER]

    bars = ax.bar(x_pos, vals, color=bar_colors, width=0.6, edgecolor="white", linewidth=0.8)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1.5, label="Random baseline (0.5)")

    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.008,
            f"{val:.3f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold",
        )

    ax.axvline(2.5, color="gray", linestyle=":", linewidth=1.0, alpha=0.6)
    ax.text(1.0, 0.38, "Supervisé",     ha="center", fontsize=9, color="#1f77b4", style="italic")
    ax.text(4.0, 0.38, "Non-supervisé", ha="center", fontsize=9, color="#2ca02c", style="italic")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(MODEL_ORDER, fontsize=11)
    ax.set_ylabel(f"{metric_label} (test set)", fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"{metric_label} — 6 modèles, baseline single-task (CWRU)",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)

mock_suffix = " [MOCK]" if IS_ANY_MOCK else ""
fig.suptitle(f"Sprint 12 — S12-09{mock_suffix}", fontsize=9, color="gray", y=0.01)
fig.tight_layout()

_path = FIGURES_DIR / "baseline_accuracy_f1_bar.png"
save_figure(fig, _path)
display(Image(str(_path)))

In [ ]:
# Section 4 — Scatter RAM peak vs. accuracy (Gap 2 — STM32 ≤ 64 Ko)

STM32_RAM_LIMIT_KB = 64.0

SCATTER_MARKERS: dict[str, tuple[str, str]] = {
    "EWC":         ("o", "#1f77b4"),
    "HDC":         ("s", "#ff7f0e"),
    "TinyOL":      ("^", "#2ca02c"),
    "KMeans":      ("D", "#d62728"),
    "Mahalanobis": ("P", "#9467bd"),
    "DBSCAN":      ("*", "#8c564b"),
}

fig, ax = plt.subplots(figsize=(8, 5))

max_ram_kb = max(r["ram_peak_bytes"] for r in results.values()) / 1024
x_max = max_ram_kb * 1.15

ax.axvspan(0, STM32_RAM_LIMIT_KB, alpha=0.08, color="green",
           label=f"Zone STM32 ≤ {STM32_RAM_LIMIT_KB:.0f} Ko")
ax.axvline(STM32_RAM_LIMIT_KB, color="red", linestyle="--", linewidth=1.5,
           label=f"Budget STM32 ({STM32_RAM_LIMIT_KB:.0f} Ko)")

for name in MODEL_ORDER:
    r = results[name]
    ram_kb = r["ram_peak_bytes"] / 1024
    acc    = r["accuracy"]
    marker, color = SCATTER_MARKERS[name]
    ax.scatter(ram_kb, acc, marker=marker, color=color, s=150, zorder=5, label=name)
    x_offset = x_max * 0.015
    ax.annotate(name, xy=(ram_kb, acc), xytext=(ram_kb + x_offset, acc + 0.003), fontsize=9)

dbscan_ram = results["DBSCAN"]["ram_peak_bytes"] / 1024
dbscan_acc = results["DBSCAN"]["accuracy"]
if dbscan_ram > STM32_RAM_LIMIT_KB:
    ax.annotate(
        f"⚠ {dbscan_ram:.0f} Ko — hors budget STM32",
        xy=(dbscan_ram, dbscan_acc),
        xytext=(dbscan_ram - x_max * 0.35, dbscan_acc + 0.05),
        fontsize=8, color="darkred",
        arrowprops=dict(arrowstyle="->", color="darkred"),
    )

ax.set_xlabel("RAM peak (Ko)", fontsize=11)
ax.set_ylabel("Accuracy (test set)", fontsize=11)
ax.set_title(
    "Trade-off embarqué : RAM vs. performance\n(baseline hors-CL — Gap 2 STM32 ≤ 64 Ko)",
    fontsize=12, fontweight="bold",
)
ax.set_xlim(0, x_max)
acc_min = min(all_accs)
ax.set_ylim(max(0.0, acc_min - 0.08), min(1.0, max(all_accs) + 0.12))
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3)
fig.tight_layout()

_path = FIGURES_DIR / "scatter_ram_vs_accuracy.png"
save_figure(fig, _path)
display(Image(str(_path)))

In [ ]:
# Section 5 — Tableau CWRU vs PRONOSTIA (mêmes 6 modèles, mêmes métriques)

PRONOSTIA_EXP_DIRS = {
    "EWC":         Path("experiments/exp_044_ewc_pronostia_no_split"),
    "HDC":         Path("experiments/exp_045_hdc_pronostia_no_split"),
    "TinyOL":      Path("experiments/exp_046_tinyol_pronostia_no_split"),
    "KMeans":      Path("experiments/exp_047_kmeans_pronostia_no_split"),
    "Mahalanobis": Path("experiments/exp_048_mahalanobis_pronostia_no_split"),
    "DBSCAN":      Path("experiments/exp_049_dbscan_pronostia_no_split"),
}
PRONOSTIA_MOCK: dict[str, dict] = {
    "EWC":         {"accuracy": 0.9602, "f1": 0.7581, "ram_peak_bytes": 1171},
    "HDC":         {"accuracy": 0.6361, "f1": 0.2846, "ram_peak_bytes": 14848},
    "TinyOL":      {"accuracy": 0.9495, "f1": 0.6696, "ram_peak_bytes": 944},
    "KMeans":      {"accuracy": 0.9004, "f1": 0.3750, "ram_peak_bytes": 5702},
    "Mahalanobis": {"accuracy": 0.8871, "f1": 0.2797, "ram_peak_bytes": 1799},
    "DBSCAN":      {"accuracy": 0.8851, "f1": 0.2575, "ram_peak_bytes": 124416},
}

pronostia_results: dict[str, dict] = {}
for name in MODEL_ORDER:
    ppath = PRONOSTIA_EXP_DIRS[name] / "results" / "metrics_single_task.json"
    if ppath.exists():
        with open(ppath) as f:
            pdata = json.load(f)
        if "accuracy" not in pdata and "acc_final" in pdata:
            pdata["accuracy"] = pdata["acc_final"]
        if "f1" not in pdata and "f1_score" in pdata:
            pdata["f1"] = pdata["f1_score"]
        pronostia_results[name] = pdata
    else:
        pronostia_results[name] = PRONOSTIA_MOCK[name].copy()

# Tableau comparatif CWRU vs PRONOSTIA
comp_rows = []
for name in MODEL_ORDER:
    cwru_r  = results[name]
    pron_r  = pronostia_results[name]
    family  = "Supervisé" if name in SUPERVISED_MODELS else "Non-supervisé"
    comp_rows.append({
        "Modèle":                name,
        "Famille":               family,
        "CWRU Acc":              f"{cwru_r.get('accuracy', 0):.3f}",
        "PRON Acc":              f"{pron_r.get('accuracy', 0):.3f}",
        "CWRU F1":               f"{cwru_r.get('f1', 0):.3f}",
        "PRON F1":               f"{pron_r.get('f1', 0):.3f}",
        "CWRU RAM (Ko)":         f"{cwru_r.get('ram_peak_bytes', 0)/1024:.1f}",
        "PRON RAM (Ko)":         f"{pron_r.get('ram_peak_bytes', 0)/1024:.1f}",
        "Δ Acc (CWRU−PRON)":     f"{cwru_r.get('accuracy',0) - pron_r.get('accuracy',0):+.3f}",
    })

df_comp = pd.DataFrame(comp_rows).set_index("Modèle")
display(Markdown("### Tableau comparatif — CWRU vs PRONOSTIA (baseline single-task)"))
display(df_comp)

# Figure PNG du tableau
fig_comp, ax_comp = plt.subplots(figsize=(13, 3.2))
ax_comp.axis("off")

col_labels = ["Modèle", "Famille",
              "CWRU Acc", "PRON Acc",
              "CWRU F1",  "PRON F1",
              "CWRU RAM Ko", "PRON RAM Ko",
              "Δ Acc"]
cell_text = []
for row in comp_rows:
    cell_text.append([
        row["Modèle"], row["Famille"],
        row["CWRU Acc"], row["PRON Acc"],
        row["CWRU F1"],  row["PRON F1"],
        row["CWRU RAM (Ko)"], row["PRON RAM (Ko)"],
        row["Δ Acc (CWRU−PRON)"],
    ])

# Couleurs de ligne alternées
row_colors = [["#f0f0f0" if i % 2 == 0 else "white"] * len(col_labels)
              for i in range(len(cell_text))]

tbl = ax_comp.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    cellColours=row_colors,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 1.6)

# En-têtes en gras
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor("#2c5f8a")
    tbl[(0, j)].set_text_props(color="white", fontweight="bold")

# Colorier Δ Acc (positif = vert, négatif = rouge)
for i, row in enumerate(comp_rows):
    delta_val = float(row["Δ Acc (CWRU−PRON)"])
    color = "#c8e6c9" if delta_val > 0 else "#ffcdd2" if delta_val < 0 else "white"
    tbl[(i + 1, 8)].set_facecolor(color)

ax_comp.set_title(
    "CWRU vs PRONOSTIA — Baseline Single-Task (mêmes 6 modèles)",
    fontsize=11, fontweight="bold", pad=12,
)
fig_comp.tight_layout()

_path = FIGURES_DIR / "cwru_vs_pronostia_comparison.png"
save_figure(fig_comp, _path)
display(Image(str(_path)))

In [ ]:
# Section 6 — Conclusion : modèle(s) à privilégier pour les scénarios CL

best_model    = max(MODEL_ORDER, key=lambda n: results[n]["accuracy"])
best_sup      = max(SUPERVISED_MODELS,   key=lambda n: results[n]["accuracy"])
best_unsup    = max(UNSUPERVISED_MODELS, key=lambda n: results[n]["accuracy"])
gap_sup_unsup = results[best_sup]["accuracy"] - results[best_unsup]["accuracy"]
models_in_budget = [n for n in MODEL_ORDER if results[n]["ram_peak_bytes"] <= 65_536]
best_lat  = min(MODEL_ORDER, key=lambda n: results[n]["inference_latency_ms"])
worst_lat = max(MODEL_ORDER, key=lambda n: results[n]["inference_latency_ms"])

# Modèles PRONOSTIA dans le budget
pron_in_budget = [n for n in MODEL_ORDER
                  if pronostia_results[n].get("ram_peak_bytes", 0) <= 65_536]

mock_note = "\n> ⚠️ *Ces résultats sont basés sur des données mock — interpréter avec précaution.*" if IS_ANY_MOCK else ""

# Écart supervisés / non-supervisés CWRU vs PRONOSTIA
cwru_unsup_mean = np.mean([results[n]["accuracy"] for n in UNSUPERVISED_MODELS])
pron_unsup_mean = np.mean([pronostia_results[n].get("accuracy", 0) for n in UNSUPERVISED_MODELS])

discussion = f"""
## Conclusion — Baseline Single-Task (Dataset 4 CWRU){mock_note}

### 1. Performance sans CL — borne supérieure
Le modèle le plus performant sans contrainte CL est **{best_model}**
(accuracy = {results[best_model]['accuracy']:.3f}, F1 = {results[best_model]['f1']:.3f}).
C'est la **borne supérieure** pour tout scénario CL sur CWRU.

### 2. Écart supervisé / non-supervisé — fort sur CWRU
Le meilleur modèle supervisé ({best_sup}, acc = {results[best_sup]['accuracy']:.3f})
dépasse le meilleur modèle non-supervisé ({best_unsup}, acc = {results[best_unsup]['accuracy']:.3f})
de **{gap_sup_unsup:.3f} points**. La moyenne des non-supervisés sur CWRU est de
{cwru_unsup_mean:.3f}, contre {pron_unsup_mean:.3f} sur PRONOSTIA — l'écart est structurel :
CWRU est multiclasse (10+ classes de défauts) alors que PRONOSTIA est binaire.

### 3. Contrainte RAM (Gap 2)
Modèles dans le budget STM32 ≤ 64 Ko : {', '.join(models_in_budget)}.
DBSCAN dépasse le budget ({results['DBSCAN']['ram_peak_bytes']/1024:.0f} Ko).
Le plus frugal est **TinyOL** ({results['TinyOL']['ram_peak_bytes']/1024:.1f} Ko),
aussi le plus rapide (latence = {results[best_lat]['inference_latency_ms']:.3f} ms).

### 4. Recommandation pour les scénarios CL
- **EWC** : priorité 1 — meilleure accuracy (≈ {results['EWC']['accuracy']:.3f}),
  RAM < 2 Ko, référence régularisation CL.
- **TinyOL** : priorité 2 — alternative architecturale embarquée,
  RAM < 1 Ko, latence minimale.
- **HDC** : priorité 3 — représentation compacte, bon F1
  ({results['HDC']['f1']:.3f}), robuste au by_fault_type.
- **KMeans / Mahalanobis / DBSCAN** : non-supervisés trop limités sur CWRU
  (acc ≈ 14–16%) pour constituer une référence CL crédible.

### 5. Pont vers les scénarios CL
Ces scores constituent la référence absolue. Les notebooks `by_fault_type/comparison.ipynb`
(exp_074–079) et `by_severity/comparison.ipynb` (exp_080–085) quantifient le **coût du CL**
(oubli catastrophique, dégradation d'accuracy) par rapport à ces baselines.

> `FIXME(gap1)` : Les non-supervisés sur CWRU sont en dessous de la chance pour Mahalanobis
> et DBSCAN en accuracy. Vérifier la configuration de détection / seuil de décision
> dans les scripts correspondants si les résultats semblent anormalement bas.
"""

display(Markdown(discussion))